# Demo 1 — Setup

This notebook prepares the Unity Catalog and Azure storage objects required by the crypto demo.

It will:
- load the shared configuration
- create or confirm the schema
- create the external Volume
- create the planned folder structure
- validate that the Volume and folders are accessible

> This notebook is designed to be safely rerunnable.

## 1. Load shared configuration

The setup notebook reuses all values from `config/00_config.ipynb`.

In [0]:
%run ../config/00_config

## 2. Display the planned setup

This section confirms the catalog, schema, Volume name, and Azure location before creating anything.

In [0]:
print("Planned setup")
print("-------------")
print(f"Catalog: {catalog}")
print(f"Schema: {schema}")
print(f"Volume: {catalog}.{schema}.{volume_name}")
print(f"Azure location: {external_volume_location}")
print(f"Volume path: {volume_root}")

## 3. Create or confirm the schema

`IF NOT EXISTS` makes the command idempotent. Rerunning it will not recreate or damage an existing schema.

In [0]:
spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}"
)

print(f"Schema ready: {catalog}.{schema}")

## 4. Validate the external location

The external location must already exist and must cover the Azure path used by this project.

This notebook does not create the storage credential or external location because those are infrastructure-level objects.

In [0]:
external_locations_df = spark.sql("SHOW EXTERNAL LOCATIONS")

matching_external_location = (
    external_locations_df
    .filter(f"name = '{external_location_name}'")
)

if matching_external_location.count() == 0:
    raise RuntimeError(
        f"External location not found: {external_location_name}"
    )

display(matching_external_location)

## 5. Create the external Volume

The Volume points to:

`abfss://parvinbadalov@dlspl21databricks.dfs.core.windows.net/demos/demo1_crypto`

The Unity Catalog name is:

`dbr_dev.parvinbadalov.demo1_crypto`

In [0]:
spark.sql(
    f"""
    CREATE EXTERNAL VOLUME IF NOT EXISTS
    {catalog}.{schema}.{volume_name}
    LOCATION '{external_volume_location}'
    """
)

print(
    f"External Volume ready: "
    f"{catalog}.{schema}.{volume_name}"
)

## 6. Create the planned folder structure

The folders are created inside the external Volume.

Planned structure:

```text
demo1_crypto/
├── raw/
│   ├── historical/
│   └── streaming_test/
├── landing/
│   └── historical/
├── system/
│   ├── schema/
│   │   └── crypto_ticks/
│   └── checkpoints/
│       └── crypto_ticks/
└── archive/
```

In [0]:
required_directories = [
    raw_path,
    historical_raw_path,
    streaming_test_path,
    landing_path,
    historical_landing_path,
    system_path,
    schema_root_path,
    crypto_ticks_schema_path,
    checkpoint_root_path,
    crypto_ticks_checkpoint_path,
    archive_path,
]

for path in required_directories:
    dbutils.fs.mkdirs(path)
    print(f"Ready: {path}")

## 7. Create readiness markers

Small `_READY` files make it easy to confirm that important folders are writable.

In [0]:
ready_markers = [
    f"{historical_raw_path}/_READY",
    f"{historical_landing_path}/_READY",
    f"{streaming_test_path}/_READY",
]

for marker_path in ready_markers:
    dbutils.fs.put(
        marker_path,
        "Demo 1 folder is ready.\n",
        overwrite=True,
    )
    print(f"Marker written: {marker_path}")

## 8. Validate the Volume

The following checks confirm that the external Volume exists and that Databricks can list its contents.

In [0]:
volume_exists = spark.sql(
    f"SHOW VOLUMES IN {catalog}.{schema}"
).filter(
    f"volume_name = '{volume_name}'"
).count() > 0

if not volume_exists:
    raise RuntimeError(
        f"Volume was not created: {catalog}.{schema}.{volume_name}"
    )

print("Volume validation passed.")
display(dbutils.fs.ls(volume_root))

## 9. Validate all required folders

Every configured folder should now exist and be accessible.

In [0]:
folder_validation = []

for path in required_directories:
    try:
        dbutils.fs.ls(path)
        folder_validation.append(
            {
                "path": path,
                "status": "READY",
            }
        )
    except Exception as exc:
        folder_validation.append(
            {
                "path": path,
                "status": f"FAILED: {exc}",
            }
        )

folder_validation_df = spark.createDataFrame(folder_validation)
display(folder_validation_df)

## 10. Final setup summary

After this notebook succeeds, upload the three Binance CSV files into:

`/Volumes/dbr_dev/parvinbadalov/demo1_crypto/raw/historical/`

In [0]:
failed_paths = [
    row["path"]
    for row in folder_validation
    if row["status"] != "READY"
]

if failed_paths:
    raise RuntimeError(
        f"Setup completed with inaccessible paths: {failed_paths}"
    )

print("Demo 1 setup completed successfully.")
print()
print(f"External Volume: {catalog}.{schema}.{volume_name}")
print(f"Azure location: {external_volume_location}")
print(f"Historical upload folder: {historical_raw_path}")
print()
print("Next step:")
print("Upload BTCUSDT, ETHUSDT, and SOLUSDT January 2026 CSV files.")